In [ ]:
# clean the data 

data = {}

for home, path in FILES.items():
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    df = df[["date", "time", "location", "event"]]
    df = df.dropna().drop_duplicates()

    for col in ["date", "time", "location", "event"]:
        df[col] = df[col].astype(str).str.strip()

    data[home] = df

In [ ]:
# combine time+stamp

for home, df in data.items():
    df["timestamp"] = pd.to_datetime(
        df["date"] + " " + df["time"], errors="coerce"
    )
    data[home] = df.dropna(subset=["timestamp"]).sort_values("timestamp")

In [ ]:
# 5 second window 
for home, df in data.items():
    df["window"] = df["timestamp"].dt.floor("5s")
    data[home] = df

In [ ]:
# feature extraction

rooms = [
    "Bedroom", "Bathroom", "DiningRoom", "GuestRoom", "Kitchen",
    "LivingRoom", "LoungeChair", "OtherRoom", "OutsideDoor",
    "WorkArea", "Hall"
]

features = {}

for home, df in data.items():
    rows = []

    for window, group in df.groupby("window"):
        row = {"timestamp": window}

        for room in rooms:
            row[f"{room}_Activity"] = (
                (group["location"] == room) &
                (group["event"].str.upper() == "ON")
            ).sum()

        row["OutsideDoor_Open"] = (
            (group["event"].str.upper() == "OPEN") &
            group["location"].str.contains("Door", case=False, na=False)
        ).sum()

        row["OutsideDoor_Close"] = (
            (group["event"].str.upper() == "CLOSE") &
            group["location"].str.contains("Door", case=False, na=False)
        ).sum()

        row["Total_Activity"] = (
            group["event"].str.upper() == "ON"
        ).sum()

        row["Total_Contact_Events"] = (
            group["event"].str.upper().isin(["OPEN", "CLOSE"])
        ).sum()

        rows.append(row)

    features[home] = pd.DataFrame(rows)

In [ ]:
for home, df in features.items():
    hour = df["timestamp"].dt.hour

    df["Time_Of_Day"] = pd.cut(
        hour,
        bins=[-1, 5, 11, 17, 23],
        labels=["Night", "Morning", "Afternoon", "Evening"]
    ).astype(str)

    df["Day_Of_Week"] = df["timestamp"].dt.day_name()
    df["Is_Night"] = hour.isin([0, 1, 2, 3, 4, 5, 22, 23])

In [ ]:
# create categories based on the count 
def categorize(value):
    if value == 0:
        return "None"
    elif value <= 2:
        return "Low"
    elif value <= 5:
        return "Medium"
    return "High"

for home, df in features.items():
    columns = [
        col for col in df.columns
        if col.endswith("_Activity") or
        col in [
            "OutsideDoor_Open", "OutsideDoor_Close",
            "Total_Activity", "Total_Contact_Events"
        ]
    ]

    for col in columns:
        df[col] = df[col].apply(categorize)

In [ ]:
# behaviour label generation

for home, df in features.items():
    active = df["Total_Activity"].isin(["Low", "Medium", "High"])
    opened = df["OutsideDoor_Open"].isin(["Low", "Medium", "High"])
    closed = df["OutsideDoor_Close"].isin(["Low", "Medium", "High"])

    df["Home_Occupancy"] = active.map({
        True: "Occupied", False: "Possibly_Empty"
    })

    df["Resident_Activity"] = active.map({
        True: "Active", False: "Inactive"
    })

    df["Resident_Sleep"] = (
        df["Is_Night"] & ~active
    ).map({
        True: "Likely_Sleeping", False: "Not_Sleeping"
    })

    df["Leaving_Home"] = (opened & ~active).map({
        True: "Likely_Leaving", False: "Unclear"
    })

    df["Returning_Home"] = (closed & active).map({
        True: "Likely_Returning", False: "Unclear"
    })

    df["Unusual_Activity"] = df["Total_Activity"].eq("High").map({
        True: "Unusual", False: "Normal"
    })

In [ ]:
train, validation, test = [], [], []

for home, df in datasets.items():
    df = df.sort_values("timestamp")
    n = len(df)

    train.append(df.iloc[:int(0.70 * n)])
    validation.append(df.iloc[int(0.70 * n):int(0.85 * n)])
    test.append(df.iloc[int(0.85 * n):])

train = pd.concat(train, ignore_index=True)
validation = pd.concat(validation, ignore_index=True)
test = pd.concat(test, ignore_index=True)


In [ ]:
# save the csv files
output = Path("processed_data")
output.mkdir(exist_ok=True)

for home, df in datasets.items():
    df.to_csv(output / f"{home.lower()}_processed.csv", index=False)

train.to_csv(output / "train.csv", index=False)
validation.to_csv(output / "validation.csv", index=False)
test.to_csv(output / "test.csv", index=False)

print("Preprocessing completed!")